# Clon de ElevenLabs — backend en Kaggle (2× T4)

Ejecuta las celdas **en orden, de arriba abajo**. Al final obtienes una URL
`https://....trycloudflare.com`: **cópiala y pégala en Settings de la app**
(frontend en GitHub Pages).

**Antes de empezar (en Kaggle, desde el teléfono):**
1. Abre este notebook y toca **⋮ → Accelerator → GPU T4 x2**.
2. En el panel derecho, **Settings → Internet: ON**.
3. Tu cuenta de Kaggle debe estar verificada por teléfono (requisito de GPU).
4. La sesión dura **12 h como máximo** y la URL del túnel **cambia en cada
   arranque**: hay que copiarla de nuevo cada vez.

Opcional: en **Add-ons → Secrets** añade `API_TOKEN` (un texto secreto
inventado por ti) para que la API exija token. Si no lo pones, la celda 5
genera uno aleatorio y lo imprime: la API YA NO queda abierta en Kaggle
(A17). En cualquier caso, copia el token en Ajustes de la app.


In [ ]:
# Celda 1 — Comprueba GPU y Python
import platform, sys
print("Python:", platform.python_version())
try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPUs:", torch.cuda.device_count(), "-", torch.cuda.get_device_name(0))
except ImportError:
    print("torch NO instalado (Kaggle normalmente ya lo trae).")


In [ ]:
# Celda 2 — Trae el código del backend
# Vía A (recomendada): el repo ya está en GitHub y contiene la carpeta backend/.
# Vía B: sube la carpeta backend/ como dataset de Kaggle llamado
#        "elevenlabs-backend" (Add data → Upload) y se usa esa copia.
import os, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get("REPO_URL", "https://github.com/autonomouswealth2026-ux/sonora.git")
BACKEND_DIR = None
dataset_dir = Path("/kaggle/input/elevenlabs-backend")
if dataset_dir.exists() and (dataset_dir / "app" / "main.py").exists():
    BACKEND_DIR = dataset_dir
    print("Backend desde dataset de Kaggle:", BACKEND_DIR)
elif (Path("/kaggle/working/backend/app/main.py").exists()):
    BACKEND_DIR = Path("/kaggle/working/backend")
    print("Backend ya presente en /kaggle/working/backend")
else:
    print("Clonando el repo:", REPO_URL)
    r = subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/sonora"],
                       capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
    cand = Path("/kaggle/working/sonora/backend")
    if (cand / "app" / "main.py").exists():
        BACKEND_DIR = cand
        print("Backend clonado en:", BACKEND_DIR)
    else:
        raise SystemExit(
            "NO se encontró backend/app/main.py. Sube la carpeta backend/ "
            "como dataset 'elevenlabs-backend' o haz push del repo a GitHub "
            "y vuelve a ejecutar esta celda."
        )
os.environ["BACKEND_DIR"] = str(BACKEND_DIR)
print("BACKEND_DIR =", BACKEND_DIR)


In [ ]:
# Celda 3 — Python 3.12 + dependencias (tarda varios minutos)
# Kaggle usa Python 3.13.16, pero kokoro/misaki exigen Python <3.13
# (sin wheel para 3.13; kokoro 0.7.16 pide misaki>=0.7.16, que no
# existe para 3.13). Se crea un venv con Python 3.12 (vía uv) y TODO
# el backend corre ahí, con los pins verificados en el venv sonora.
# OJO: `uv venv` NO trae pip dentro del venv: las instalaciones se
# hacen con `uv pip install --python <venv>`, nunca `venv -m pip`.
import os, subprocess, sys
from pathlib import Path
backend = Path(os.environ["BACKEND_DIR"])
venv = backend / ".venv-gpu"
venv_py = venv / "bin" / "python"
print("Espacio en disco:")
subprocess.run(["df", "-h", "/kaggle/working"])
print("Instalando espeak-ng y ffmpeg (necesarios para Kokoro)...")
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-qq", "-y", "espeak-ng", "ffmpeg"], check=False)
print("Instalando uv y Python 3.12...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=False)
r = subprocess.run(["uv", "python", "install", "3.12"], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
if not venv_py.exists():
    r = subprocess.run(["uv", "venv", str(venv), "--python", "3.12"], capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
r = subprocess.run([str(venv_py), "-c", "import sys; print(sys.version)"], capture_output=True, text=True)
print(r.stdout, r.stderr)
print("Instalando torch con CUDA (cu130) en el venv...")
r = subprocess.run(["uv", "pip", "install", "--python", str(venv_py), "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cu130"],
                   capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-1000:])
print("Instalando requirements-gpu.txt en el venv...")
r = subprocess.run(["uv", "pip", "install", "--python", str(venv_py), "-q", "-r",
                    str(backend / "requirements-gpu.txt")],
                   capture_output=True, text=True)
print(r.stdout[-1000:])
print(r.stderr[-2000:])
print("Return code:", r.returncode)
if r.returncode != 0:
    raise SystemExit("FALLO la instalación de dependencias: revisa el error arriba.")
print("Verificando imports en el venv...")
r = subprocess.run([str(venv_py), "-c",
    "import torch, kokoro, faster_whisper, av, demucs; "
    "print('torch', torch.__version__, 'cuda:', torch.cuda.is_available()); "
    "print('kokoro OK, av', av.__version__)"], capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
if r.returncode != 0:
    raise SystemExit("FALLO la verificación de imports.")
os.environ["VENV_PY"] = str(venv_py)
print("VENV_PY =", venv_py)


In [ ]:
# Celda 4 — Descarga los pesos (Qwen3-TTS + Kokoro + Whisper)
# Se descargan con el Python del venv (VENV_PY), que es el que tiene
# huggingface_hub instalado. La primera vez tarda varios minutos
# (varios GB); Kaggle los guarda en caché entre sesiones.
# OJO: `huggingface-cli download` YA NO EXISTE (el CLI ahora es `hf`);
# se usa la API snapshot_download, que además verifica cada descarga.
import os, subprocess
MODELS = [
    "Qwen/Qwen3-TTS-Tokenizer-12Hz",
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    "hexgrad/Kokoro-82M",
]
# Variante secundaria: si falla o no cabe en disco, se sigue sin ella.
OPTIONAL = ["Qwen/Qwen3-TTS-12Hz-0.6B-Base"]
venv_py = os.environ["VENV_PY"]
ok = True
for repo in MODELS + OPTIONAL:
    optional = repo in OPTIONAL
    print("Descargando", repo, "...")
    r = subprocess.run([venv_py, "-c",
        "from huggingface_hub import snapshot_download; "
        f"print(snapshot_download(repo_id='{repo}'))"],
        capture_output=True, text=True)
    if r.returncode == 0:
        print("  -> OK:", r.stdout.strip().splitlines()[-1])
    else:
        print("  -> FALLO:", r.stderr[-500:])
        if not optional:
            ok = False
print("Descargando Whisper large-v3 (faster-whisper)...")
r = subprocess.run([venv_py, "-c",
    "from faster_whisper import WhisperModel; "
    "m = WhisperModel('large-v3', device='cuda', compute_type='float16'); "
    "del m; print('Whisper large-v3 listo')"],
    capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
if r.returncode != 0:
    ok = False
if not ok:
    raise SystemExit("FALLO la descarga de modelos: revisa los errores "
                     "arriba y re-ejecuta esta celda.")
print("Pesos listos (verificado).")


In [ ]:
# Celda 5 — Arranca el servidor FastAPI en segundo plano
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path

backend = Path(os.environ["BACKEND_DIR"])
venv_py = os.environ.get("VENV_PY")
if not venv_py:
    raise SystemExit("VENV_PY no está definido: ejecuta la celda 3 primero.")
log = open(backend / "server.log", "wb")
env = dict(os.environ)
env["PORT"] = "8000"
# B6: CORS para el frontend en GitHub Pages. Sin esto el navegador
# bloquea todas las llamadas (Pages y trycloudflare son orígenes
# distintos). Se puede sobreescribir con CORS_ORIGINS en Secrets.
env["CORS_ORIGINS"] = os.environ.get(
    "CORS_ORIGINS",
    "https://autonomouswealth2026-ux.github.io",
)
# A17: en Kaggle el token es OBLIGATORIO. Si no viene en Secrets, se
# genera uno aleatorio y se imprime: hay que copiarlo en Ajustes.
import secrets
if not env.get("API_TOKEN"):
    env["API_TOKEN"] = secrets.token_urlsafe(24)
    print("API_TOKEN generado (la API exige token en Kaggle):")
    print(" ", env["API_TOKEN"])
    print("Copialo en Ajustes de la app, campo Token.")
else:
    print("API_TOKEN tomado de Secrets de Kaggle.")
proc = subprocess.Popen(
    [venv_py, "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(backend), stdout=log, stderr=subprocess.STDOUT,
    env=env, start_new_session=True,
)
print("Servidor lanzado, PID:", proc.pid)
print("Esperando a /health ...")
ok = False
for i in range(60):
    time.sleep(2)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as resp:
            print("HEALTH:", json.loads(resp.read()))
            ok = True
            break
    except Exception:
        pass
if not ok:
    print("El servidor no respondió. Mira el final de server.log:")
    print((backend / "server.log").read_text()[-3000:])
else:
    print("Servidor UP. Ningún modelo cargado todavía (carga perezosa).")


In [ ]:
# Celda 6 — Túnel trycloudflare: imprime la URL https final
# *** COPIA LA URL QUE IMPRIME ESTA CELDA Y PÉGALA EN SETTINGS DE LA APP ***
import re, subprocess, time
from pathlib import Path

backend = Path(os.environ["BACKEND_DIR"])
cf = backend / "cloudflared"
if not cf.exists():
    print("Descargando cloudflared...")
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", str(cf)], check=False)
    cf.chmod(0o755)
log = open(backend / "cloudflared.log", "wb")
proc = subprocess.Popen(
    [str(cf), "tunnel", "--url", "http://localhost:8000"],
    stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
)
print("Túnel lanzado, PID:", proc.pid)
url = None
for i in range(45):
    time.sleep(2)
    text = (backend / "cloudflared.log").read_text(errors="ignore")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if m:
        url = m.group(0)
        break
if url:
    print()
    print("=" * 60)
    print("  URL DEL BACKEND (copiala en Settings de la app):")
    print(" ", url)
    print("=" * 60)
    print()
    print("Comprueba en el navegador:", url + "/health")
    print("La sesión dura 12 h. No cierres esta pestaña de Kaggle.")
else:
    print("No apareció la URL. Últimas líneas de cloudflared.log:")
    print((backend / "cloudflared.log").read_text(errors="ignore")[-2000:])


## Si algo falla

- **`/health` no responde**: abre `server.log` (en la carpeta del backend,
  panel derecho de Kaggle → Files) y mira el error al final.
- **La URL no aparece**: espera 1 minuto y vuelve a ejecutar solo la celda 6.
- **`torch` no instalado**: en una celda nueva ejecuta
  `!pip install torch --index-url https://download.pytorch.org/whl/cu124`
  y repite desde la celda 3.
- **Error de cuota GPU (30 h/semana)**: espera al reset semanal de Kaggle.

## Alternativa: ngrok (URL fija, con cuenta gratuita)

Si te molesta copiar la URL en cada sesión: crea una cuenta en ngrok, copia
tu authtoken en Secrets de Kaggle como `NGROK_TOKEN`, y en vez de la celda 6
ejecuta en una celda nueva:

```python
import subprocess, time
subprocess.run(["pip", "install", "-q", "pyngrok"], check=True)
from pyngrok import ngrok
import os
ngrok.set_auth_token(os.environ["NGROK_TOKEN"])
tunnel = ngrok.connect(8000)
print("URL fija:", tunnel.public_url)
```

Ojo: el plan gratis de ngrok tiene 1 GB de transferencia al mes.

## Para parar

Detén el notebook (Run → Interrupt / apaga la sesión). La URL muere con la
sesión: la próxima vez hay que repetir desde la celda 5.
